In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [16]:
# Average Rating vs Watches by top 20 Languages (Side-by-Side)

import pandas as pd
import altair as alt

lang_df = pd.read_csv("Movie_Data_File.csv").copy()
lang_df = lang_df.dropna(subset=["Original_language", "Watches", "Average_rating"])
lang_df = lang_df[lang_df["Original_language"] != "No spoken language"]
lang_df = lang_df[lang_df["Original_language"] != "English"]   # non-English only
lang_df = lang_df[lang_df["Watches"] > 0]

# Aggregate per language
lang_agg = lang_df.groupby("Original_language").agg(
    total_watches=("Watches", "sum"),
    avg_rating=("Average_rating", "mean"),
    film_count=("Watches", "count")
).reset_index()

# Top 20 non-English languages by film count
lang_agg = lang_agg.nlargest(20, "film_count").reset_index(drop=True)

# Convert to millions
lang_agg["watches_millions"] = (lang_agg["total_watches"] / 1_000_000).round(2)

# text mark — language name as the data point
chart = alt.Chart(lang_agg).mark_text(fontSize=12, fontWeight="bold").encode(
    x=alt.X("avg_rating:Q", title="Average Rating",
            scale=alt.Scale(domain=[3, 4])),
    y=alt.Y("watches_millions:Q", title="Total Watches (Millions)"),
    text=alt.Text("Original_language:N"),
    color=alt.value("#1c2b3a"),
    tooltip=[
        alt.Tooltip("Original_language:N", title="Language"),
        alt.Tooltip("avg_rating:Q",        title="Avg Rating",  format=".2f"),
        alt.Tooltip("watches_millions:Q",  title="Watches (M)", format=".1f"),
        alt.Tooltip("film_count:Q",        title="Number of Films")
    ]
).properties(
    title="Top 20 Non-English Languages: Avg Rating vs Total Watches",
    width=620,
    height=420
)

chart.save("images/language_rating_scatter.html")
chart.show()

alt.Chart(...)

In [ ]:
# Top 10 Films by Language
import pandas as pd
import ast
import altair as alt

films_df = pd.read_csv("Movie_Data_File.csv").copy()
films_df = films_df.dropna(subset=["Original_language", "Average_rating", "Film_title", "Genres"])
films_df = films_df[films_df["Original_language"] != "No spoken language"]
films_df = films_df[films_df["Original_language"] != "English"]   # non-English only

# Parse and explode genres — strip any stray [ or ' characters
films_df["Genres"] = films_df["Genres"].apply(ast.literal_eval)
films_df = films_df.explode("Genres")
films_df["Genres"] = films_df["Genres"].str.strip().str.title()

# keep only the top 20 non-English languages by film count
top20_langs = (
    films_df.groupby("Original_language")["Film_title"]
    .count()
    .nlargest(20)
    .index
    .tolist()
)
films_df = films_df[films_df["Original_language"].isin(top20_langs)]

# keep only major genres
MAJOR_GENRES = {
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "Thriller", "War", "Western"
}
films_df = films_df[films_df["Genres"].isin(MAJOR_GENRES)]

languages = sorted(top20_langs)
genres    = ["All"] + sorted(MAJOR_GENRES)

rows = []
for lang in languages:
    lang_films = films_df[films_df["Original_language"] == lang]

    # all genres combined
    top10_all = (
        lang_films.groupby("Film_title").agg(
            Average_rating=("Average_rating", "mean")
        ).reset_index()
        .sort_values("Average_rating", ascending=False)
        .head(10).reset_index(drop=True)
    )
    for i, r in top10_all.iterrows():
        rows.append({
            "Film_title":        r["Film_title"],
            "Original_language": lang,
            "Genre":             "All",
            "Average_rating":    round(r["Average_rating"], 2),
            "rank":              i + 1
        })

    # per genre
    for genre in lang_films["Genres"].unique():
        top10_genre = (
            lang_films[lang_films["Genres"] == genre]
            .groupby("Film_title").agg(
                Average_rating=("Average_rating", "mean")
            ).reset_index()
            .sort_values("Average_rating", ascending=False)
            .head(10).reset_index(drop=True)
        )
        for i, r in top10_genre.iterrows():
            rows.append({
                "Film_title":        r["Film_title"],
                "Original_language": lang,
                "Genre":             genre,
                "Average_rating":    round(r["Average_rating"], 2),
                "rank":              i + 1
            })

prebuilt = pd.DataFrame(rows)

# dropdowns
language_param = alt.param(
    name="language_val",
    bind=alt.binding_select(options=languages, name="Language: "),
    value=languages[0]
)
genre_param = alt.param(
    name="genre_val",
    bind=alt.binding_select(options=genres, name="Genre: "),
    value="All"
)

base = alt.Chart(prebuilt).transform_filter(
    "datum.Original_language === language_val"
).transform_filter(
    "datum.Genre === genre_val"
)

bars = base.mark_bar(color="#185FA5").encode(
    x=alt.X("Average_rating:Q", title="Average Rating",
            scale=alt.Scale(domain=[0, 5])),
    y=alt.Y("rank:O", title=None, axis=None,
            scale=alt.Scale(paddingOuter=0.2, paddingInner=0.3)),
    tooltip=[
        alt.Tooltip("Film_title:N",     title="Film"),
        alt.Tooltip("Genre:N",          title="Genre"),
        alt.Tooltip("Average_rating:Q", title="Rating", format=".2f"),
    ]
)

film_labels = base.mark_text(
    align="left", dx=6, fontSize=12, color="white", fontWeight="bold"
).encode(
    x=alt.value(0),
    y=alt.Y("rank:O", axis=None,
            scale=alt.Scale(paddingOuter=0.2, paddingInner=0.3)),
    text=alt.Text("Film_title:N")
)

rating_labels = base.mark_text(
    align="left", dx=6, fontSize=12, color="#333"
).encode(
    x=alt.X("Average_rating:Q"),
    y=alt.Y("rank:O", axis=None,
            scale=alt.Scale(paddingOuter=0.2, paddingInner=0.3)),
    text=alt.Text("Average_rating:Q", format=".2f")
)

chart = (bars + film_labels + rating_labels).add_params(
    language_param,
    genre_param
).properties(
    title="Top 10 Rated Films by Language and Genre",
    width=460,
    height=400
)

chart.save("images/top_films_by_language.html")
chart.show()

alt.LayerChart(...)

: 

: 

: 

: 

: 